# Preparing spliced/unspliced data

`partition_de_by_mechanism` needs two count layers per cell — `spliced`
(mature mRNA) and `unspliced` (nascent transcript) — on top of the normal
`.X`. This notebook covers where those layers come from (four common
upstream quantifiers, with pinned versions and full commands), how to load
each tool's output into `AnnData`, how to merge them into an
already-QC'd object, and how to sanity-check the result before calling
scATrans.

This is reference material, not a run against one bundled dataset. Section 1
and 2's shell/loading commands need your own FASTQ/BAM files and reference
genome, so they are shown as code blocks but **not executed** here. Sections
3–5 *are* executed, against a small synthetic example, so you can see the
exact shapes, mismatches, and diagnostics involved.

If you already have `adata.layers["spliced"]` / `["unspliced"]` (or
`["mature"]` / `["nascent"]`), skip to {doc}`../quickstart`.

In [1]:
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc

import scatrans as scat

warnings.filterwarnings("ignore")
print("scatrans", scat.__version__)

scatrans 0.10.9


## 1. Pick an upstream quantifier

Standard scRNA-seq alignment (Cell Ranger `count`, STARsolo with only
`--soloFeatures Gene`) discards the intron/exon split needed here. You need
a tool run in a **velocity-aware mode**. Versions below are current as of
this writing (Aug 2026) — check each tool's release page before a new
project, this corner of the ecosystem moves fast.

| Tool | Version | Best when… |
|------|---------|-------------|
| **velocyto** | 0.17.17 | You already ran Cell Ranger and just want a quick loom file from its BAM |
| **STARsolo** (part of STAR) | 2.7.11b | Aligning from FASTQ and want Gene + Velocyto counts in one pass |
| **kb-python** (kallisto\|bustools) | 0.30.2, `--workflow nac` | Fast pseudoalignment-based counting, one command per sample |
| **alevin-fry** + **pyroe** | 0.11.2 / 0.9.3 | Fastest large-cohort throughput (selective alignment + USA mode) |

All four are read-alignment-based (intronic vs. exonic reads), not metabolic
labeling. If your data comes from scNT-seq, sci-fate, or another 4sU/labeling
protocol, skip to §5 ("Metabolic labeling") below — you
already have the layers you need from the demultiplexing pipeline, and
should not run any of the four tools below on labeling data.

### velocyto (0.17.17)

Runs directly on a completed `cellranger count` output folder — no separate
alignment step.

```bash
pip install velocyto==0.17.17   # last velocyto.py release (2019); still the
                                 # standard wrapper around a Cell Ranger BAM

# SAMPLE_DIR is the cellranger count output folder (contains outs/possorted_genome_bam.bam,
# outs/filtered_feature_bc_matrix/, outs/raw_feature_bc_matrix/)
velocyto run10x SAMPLE_DIR /path/to/refdata-gex-GRCh38-2024-A/genes/genes.gtf

# recommended: mask repeat regions to cut spurious intronic reads
# (download a repeat-masker GTF for your genome build first)
# velocyto run10x -m repeat_msk.gtf SAMPLE_DIR genes.gtf
```

Output: `SAMPLE_DIR/velocyto/<sample>.loom`, with `spliced` / `unspliced` /
`ambiguous` layers already inside.

### STARsolo (STAR 2.7.11b)

One alignment pass produces both the standard gene count matrix and the
velocity split.

```bash
# 1) build a genome index once per reference (skip if you already have one)
STAR --runMode genomeGenerate \
     --genomeDir STAR_index \
     --genomeFastaFiles genome.fa \
     --sjdbGTFfile annotation.gtf \
     --sjdbOverhang 90 --runThreadN 8

# 2) align + quantify, with Velocyto features on top of Gene
STAR --runMode alignReads \
     --genomeDir STAR_index \
     --readFilesIn R2.fastq.gz R1.fastq.gz \
     --readFilesCommand zcat \
     --soloType CB_UMI_Simple \
     --soloCBwhitelist 3M-february-2018.txt \
     --soloCBstart 1 --soloCBlen 16 --soloUMIstart 17 --soloUMIlen 12 \
     --soloFeatures Gene Velocyto \
     --outSAMtype BAM Unsorted \
     --runThreadN 8 \
     --outFileNamePrefix sample_
```

`3M-february-2018.txt` is the 10x v3 barcode whitelist (ships with Cell
Ranger, `cellranger-cs/*/lib/python/cellranger/barcodes/`); for v2 chemistry
use `737K-august-2016.txt` and `--soloCBlen 16 --soloUMIlen 10`.

Output: `sample_Solo.out/Velocyto/raw/{spliced,unspliced,ambiguous}.mtx`
plus matching `barcodes.tsv` / `features.tsv` in the same folder.

### kb-python (0.30.2, `--workflow nac`)

`nac` ("nascent and cDNA") replaced the older `lamanno` workflow name in
recent kb-python and writes ready-to-use layers directly into an `.h5ad`.

```bash
pip install kb-python==0.30.2

# 1) build the nac index (once per reference)
kb ref --workflow nac \
       -i index.idx -g t2g.txt \
       -f1 cdna.fa -f2 nascent.fa \
       -c1 cdna_t2c.txt -c2 nascent_t2c.txt \
       genome.fa annotation.gtf

# 2) pseudoalign + quantify one sample
kb count --workflow nac \
         -i index.idx -g t2g.txt \
         -c1 cdna_t2c.txt -c2 nascent_t2c.txt \
         -x 10xv3 -o out_dir \
         --h5ad \
         R1.fastq.gz R2.fastq.gz
```

Output: `out_dir/counts_unfiltered/adata.h5ad`, already containing
`adata.layers["mature"]`, `["nascent"]`, and `["ambiguous"]` — this is the
`mature`/`nascent` naming scATrans resolves automatically (see §2). Older
kb-python (<0.27) used `--workflow lamanno` and wrote `spliced.mtx` /
`unspliced.mtx` matrix files instead of an `.h5ad`.

### alevin-fry (0.11.2) + pyroe (0.9.3), USA mode

Fastest option for large cohorts; needs `salmon` and the `alevin-fry`
binary in addition to the `pyroe` Python package.

```bash
pip install pyroe==0.9.3
# salmon and alevin-fry are separate binaries, e.g. via conda:
# conda install -c bioconda salmon=1.10.3 alevin-fry=0.11.2

# 1) build a spliced+intron ("splici") reference — read-length is your R2 length minus
#    flank-trim-length (91 - 5 = 86 here, for a 91bp 10x v3 cDNA read)
pyroe make-splici genome.fa annotation.gtf 91 splici_ref \
     --flank-trim-length 5 --filename-prefix splici

# 2) index and map
salmon index -t splici_ref/splici_fl86.fa -i splici_idx -p 8

salmon alevin -l ISR -i splici_idx \
     -1 R1.fastq.gz -2 R2.fastq.gz \
     --chromiumV3 --sketch -p 8 -o alevin_map

# 3) resolve unspliced/spliced/ambiguous (USA) counts
alevin-fry generate-permit-list -d fw -k -i alevin_map -o af_quant
alevin-fry collate -t 8 -i af_quant -r alevin_map
alevin-fry quant -t 8 -i af_quant -o af_quant_res \
     --tg-map splici_ref/splici_fl86_t2g_3col.tsv \
     --resolution cr-like --use-mtx
```

Use `--chromium` instead of `--chromiumV3` for 10x v2 chemistry. Output:
loaded in Python with `pyroe.load_fry`, not read directly (see §2).

## 2. Load the output into AnnData

Each tool's output lands in AnnData slightly differently.

**velocyto (`.loom`)** — layers are already named `spliced` / `unspliced`:

```python
import anndata as ad

adata = ad.read_loom("SAMPLE_DIR/velocyto/sample.loom")
# adata.layers["spliced"], adata.layers["unspliced"], adata.layers["ambiguous"]
```

**STARsolo (`Velocyto/raw/*.mtx`)** — three separate Matrix Market files to stitch together:

```python
import scanpy as sc
import pandas as pd

base = "sample_Solo.out/Velocyto/raw"
spliced = sc.read_mtx(f"{base}/spliced.mtx").T
unspliced = sc.read_mtx(f"{base}/unspliced.mtx").T

genes = pd.read_csv(f"{base}/features.tsv", header=None, sep="\t")[1].values
barcodes = pd.read_csv(f"{base}/barcodes.tsv", header=None)[0].values

adata = spliced
adata.var_names = genes
adata.obs_names = barcodes
adata.layers["spliced"] = spliced.X.copy()
adata.layers["unspliced"] = unspliced.X
```

**kb-python (`--workflow nac`)** — already a ready `.h5ad`, no renaming needed
(scATrans resolves `mature`/`nascent` automatically):

```python
adata = sc.read_h5ad("out_dir/counts_unfiltered/adata.h5ad")
# adata.layers["mature"], adata.layers["nascent"], adata.layers["ambiguous"]
```

**alevin-fry (via `pyroe`)**:

```python
from pyroe import load_fry

# output_format="velocity" -> layers "spliced" and "unspliced" directly
# (there is no "scVelo" format string; "velocity" is the correct one)
adata = load_fry("af_quant_res", output_format="velocity")
```

## 3. Merge into your main (already-QC'd) AnnData

Most workflows already have a filtered, QC'd `adata` from a standard
Cell Ranger / STARsolo `Gene`-only run — for example, one you ran through
`sc.pp.filter_cells`, doublet removal, and clustering. Attach the velocity
layers to *that* object rather than restarting analysis from the raw
velocyto/kb-python/alevin-fry output, so QC and cell selection stay in the
count matrix you trust.

The example below builds two tiny synthetic AnnData objects that stand in
for `adata_main` (your QC'd object) and `adata_velocity` (a quantifier's
output) — with one deliberately introduced barcode-suffix mismatch, the most
common reason this join produces near-zero overlap in practice.

In [2]:
rng = np.random.default_rng(0)
n_cells, n_genes = 30, 15
gene_names = [f"Gene{i}" for i in range(n_genes)]

# adata_main: already QC'd, Cell Ranger multi-sample barcodes carry a "-1" suffix
barcodes_main = [f"{''.join(rng.choice(list('ACGT'), 16))}-1" for _ in range(n_cells)]
adata_main = ad.AnnData(
    X=rng.poisson(5, size=(n_cells, n_genes)).astype(float),
    obs=pd.DataFrame({"condition": rng.choice(["Control", "Disease"], n_cells)}, index=barcodes_main),
    var=pd.DataFrame(index=gene_names),
)

# adata_velocity: same cells/genes, but the quantifier's barcodes have no "-1" suffix
# (a very common real-world mismatch) and cover a slightly different gene set
barcodes_velocity = [b.replace("-1", "") for b in barcodes_main]
velocity_genes = gene_names[:-2] + ["DecoyGeneA", "DecoyGeneB"]
adata_velocity = ad.AnnData(
    X=rng.poisson(3, size=(n_cells, n_genes)).astype(float),
    obs=pd.DataFrame(index=barcodes_velocity),
    var=pd.DataFrame(index=velocity_genes),
)
adata_velocity.layers["unspliced"] = rng.poisson(1.2, size=(n_cells, n_genes)).astype(float)
adata_velocity.layers["spliced"] = adata_velocity.X.copy()

print("adata_main barcodes: ", adata_main.obs_names[:2].tolist())
print("adata_velocity barcodes:", adata_velocity.obs_names[:2].tolist())

adata_main barcodes:  ['TGGCCAAAATGTGGTG-1', 'GGGTCTGACTGATGTA-1']
adata_velocity barcodes: ['TGGCCAAAATGTGGTG', 'GGGTCTGACTGATGTA']


In [3]:
# Naive intersection BEFORE fixing the suffix mismatch
common_cells = adata_main.obs_names.intersection(adata_velocity.obs_names)
print(f"common cells without fixing barcodes: {len(common_cells)} / {n_cells}")

common cells without fixing barcodes: 0 / 30


Zero (or near-zero) overlap — exactly the silent failure mode to check
for before merging. Fix the suffix, then merge properly.

In [4]:
# Fix the suffix, then merge
adata_velocity.obs_names = [b + "-1" for b in adata_velocity.obs_names]

common_cells = adata_main.obs_names.intersection(adata_velocity.obs_names)
common_genes = adata_main.var_names.intersection(adata_velocity.var_names)
print(f"common cells after fixing barcodes: {len(common_cells)} / {n_cells}")
print(f"common genes: {len(common_genes)} / {n_genes} "
      f"(velocity file also had {sorted(set(adata_velocity.var_names) - set(gene_names))})")

adata_merged = adata_main[common_cells, common_genes].copy()
adata_velocity_aligned = adata_velocity[common_cells, common_genes]
adata_merged.layers["spliced"] = adata_velocity_aligned.layers["spliced"]
adata_merged.layers["unspliced"] = adata_velocity_aligned.layers["unspliced"]

print(adata_merged)

common cells after fixing barcodes: 30 / 30
common genes: 13 / 15 (velocity file also had ['DecoyGeneA', 'DecoyGeneB'])
AnnData object with n_obs × n_vars = 30 × 13
    obs: 'condition'
    layers: 'spliced', 'unspliced'


This manual `.obs_names.intersection(...)` / `.var_names.intersection(...)`
join is what `scv.utils.merge(adata_main, adata_velocity)` does for you (plus
a few extra scVelo-specific bookkeeping steps) — reach for it once
`pip install "scatrans[advanced]"` is installed and you'd rather not write
the intersection by hand:

```python
import scvelo as scv

scv.utils.merge(adata_main, adata_velocity)  # modifies adata_main in place
```

Other common causes of a small/empty intersection, beyond the suffix demo
above:

- **Raw vs. filtered barcode list.** velocyto/STARsolo/kb-python often output
  against the *raw* (unfiltered) barcode whitelist; your main object is
  filtered. The intersection approach above handles this correctly as long
  as you don't need cells present only in the velocity file.
- **Ambiguous counts dropped.** All four tools also emit an `ambiguous`
  layer (reads compatible with both spliced and unspliced). scATrans does
  not use it — leave it out, or add it into `spliced` if your assay has very
  few ambiguous reads and you want conservative unspliced calls. Don't add
  it to `unspliced`; that inflates the nascent signal.

## 4. Sanity-check before running scATrans

`scat.qc.regime_diagnosis` folds the global unspliced fraction into a
reliability score that `partition_de_by_mechanism` uses automatically. Run
it once here to make sure the merge above produced something sane before a
full analysis.

In [5]:
frac = scat.qc.unspliced_global(adata_merged)
r = scat.qc.regime_diagnosis(adata_merged)
print(f"global unspliced fraction: {frac:.2f}")
print(f"regime: {r['regime']}, reliability: {r['reliability']:.2f}")
print(r["message"])

global unspliced fraction: 0.30
regime: ok, reliability: 1.00
unspliced fraction 30.1% is in the normal band; proxy not obviously corrupted.


A healthy 10x 3' library typically lands around 10–45% global unspliced
fraction, where reliability is at its maximum. Above ~50–70% usually means
nuclear enrichment, gDNA contamination, or a barcode/layer mixup rather than
real biology — see the "Global unspliced fraction > 50%" entry in
{doc}`../faq`. For comparison, here is the same object with unspliced counts
scaled up to simulate that failure mode:

In [6]:
adata_bad = adata_merged.copy()
adata_bad.layers["unspliced"] = adata_bad.layers["unspliced"] * 25  # simulate gDNA/nuclear contamination

frac_bad = scat.qc.unspliced_global(adata_bad)
r_bad = scat.qc.regime_diagnosis(adata_bad)
print(f"global unspliced fraction: {frac_bad:.2f}")
print(f"regime: {r_bad['regime']}, reliability: {r_bad['reliability']:.2f}")
print(r_bad["message"])

global unspliced fraction: 0.92
regime: high_unspliced, reliability: 0.00
unspliced fraction 91.5% is high (>= 45%): possible nuclear/gDNA contamination -> gamma fit and the nascent proxy may be unreliable; mechanism annotations down-weighted.


Reliability drops sharply — `partition_de_by_mechanism` would scale
down `mechanism_confidence` accordingly rather than reporting mechanism
labels it can't back up.

## 5. Metabolic labeling (scNT-seq, sci-fate, etc.)

If your assay separates new vs. pre-existing RNA by 4sU labeling rather than
by intron/exon alignment, you likely already have `new`/`old` or
`labeled`/`unlabeled` count matrices from the demultiplexing pipeline (e.g.
the scNT-seq or sci-fate processing scripts). Rename those layers to match
what scATrans expects — no velocyto/kb-python/STARsolo/alevin-fry run
needed:

In [7]:
adata_labeling = adata_merged.copy()
adata_labeling.layers["old"] = adata_labeling.layers.pop("spliced")
adata_labeling.layers["new"] = adata_labeling.layers.pop("unspliced")

# --- the rename scATrans needs ---
adata_labeling.layers["spliced"] = adata_labeling.layers["old"]      # pre-existing / mature
adata_labeling.layers["unspliced"] = adata_labeling.layers["new"]    # newly synthesized / nascent

print(sorted(adata_labeling.layers.keys()))

['new', 'old', 'spliced', 'unspliced']


Labeling-based nascent fractions are a different (generally cleaner)
signal than intronic-read capture, but they go through the same
`regime_diagnosis` reliability check run in §4 — see
{doc}`../domain_assumptions` for how that reliability score is computed from
the unspliced fraction.

## Next

- {doc}`../quickstart` — run `partition_de_by_mechanism` once layers are in place.
- {doc}`t_gse226488_partition_mechanism` — worked example on real 10x data.
- {doc}`../faq` — troubleshooting layer/regime issues.